In [ ]:
# 加入了超参数
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from dataset_create import prepare_multiple_datasets_and_scaler, ProcessedDrivingDataset, inverse_transform
from model import DualStreamDrivingModel
from datetime import datetime

def train():
    # 配置文件路径
    input_csv = 'global_vehicle_data.csv'
    output_csv = 'global_vehicle_data_history_cols.csv'
    scaler_json = 'scaler_params.json'
    target_names = ['acceleration_x', 'acceleration_y', 'acceleration_z', 'steer']

    # 数据预处理
    print(">>> 开始检查并预处理数据...")
    scaler_params = prepare_multiple_datasets_and_scaler(
        input_file=input_csv,
        output_csv=output_csv,
        scaler_json_path=scaler_json,
        seq_length=9
    )
    print(">>> 预处理完成！\n")

    # 加载数据集
    dataset = ProcessedDrivingDataset(csv_file=output_csv)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

    # 初始化模型、损失函数、优化器
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DualStreamDrivingModel().to(device)
    optimizer = Adam(model.parameters(), lr=1e-4)
    scheduler = StepLR(optimizer, step_size=10, gamma=0.9)

    # 设置加权损失的权重超参数
    a = 0.3  # 速度的权重，转角的权重为 1-a

    # 日志文件
    log_dir = "logs"
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f"training_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")

    # 训练循环
    num_epochs = 20
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for batch_idx, (images, state_seq, target) in enumerate(dataloader):
            # 数据移至设备
            img_t_minus_2, img_t_minus_1, img_t = [img.to(device) for img in images]
            state_seq = state_seq.to(device)
            target = target.to(device)

            # 前向传播
            predictions = model(img_t_minus_2, img_t_minus_1, img_t, state_seq)

            # 计算加权损失
            acceleration_loss = nn.MSELoss()(predictions[:, :3], target[:, :3])  # 加速度损失
            steer_loss = nn.MSELoss()(predictions[:, 3], target[:, 3])           # 转角损失
            loss = a * acceleration_loss + (1 - a) * steer_loss

            # 反向传播与优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            # 打印当前 Batch 的损失
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}")

        # 学习率调度
        scheduler.step()

        # 记录日志
        with open(log_file, "a") as f:
            f.write(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(dataloader):.4f}\n")

        print(f"Epoch [{epoch+1}/{num_epochs}] 完成，平均 Loss: {epoch_loss/len(dataloader):.4f}")

    # 保存模型
    model_path = os.path.join(log_dir, "v_a0.3.pth")
    torch.save(model.state_dict(), model_path)
    print(f"模型已保存至 {model_path}")

if __name__ == "__main__":
    train()